# Dataset 3 — Demand Forecasting Kernels

**Author:** Mohd Ashraf Huzairie  
**Study:** Comparative Performance Analysis of Forecasting Models for Automated Warehouse Replenishment

This notebook uses the labeled training file for honest evaluation. Kaggle's unlabeled test sales are not replaced with artificial zero targets.

## 1. Setup and data contract

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from warehouse_forecasting.data import load_demand_series
from warehouse_forecasting.experiment import run_experiment
from warehouse_forecasting.paper_results import table as paper_table

DATA_PATH = ROOT / 'data/dataset3/train.csv'
DATASET = 'store_item'
RUN_TRAINING = False
print(f'Labeled training data available: {DATA_PATH.exists()}')

## 2. Clean and aggregate sales

Sales across store-item combinations are aggregated by date. Chronological interpolation is applied only after aggregation, and all scalers are fitted within their corresponding training fold.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Download Dataset 3 and place train.csv at {DATA_PATH.relative_to(ROOT)}')
series = load_demand_series(DATA_PATH, DATASET)
display(series.describe().to_frame('daily_sales'))
print(f'Date range: {series.index.min().date()} to {series.index.max().date()}')
print(f'Missing values after cleaning: {series.isna().sum()}')

## 3. Compact exploratory analysis

In [ ]:
ax = series.plot(figsize=(12, 4), alpha=.4, label='Daily sales')
series.rolling(30, min_periods=1).mean().plot(ax=ax, linewidth=2, label='30-day mean')
ax.set(title='Dataset 3 aggregate store-item demand', xlabel='Date', ylabel='Sales')
ax.legend(); plt.tight_layout()

## 4. Time-aware evaluation

Evaluation uses only known targets from `train.csv` and expanding-window folds. The competition `test.csv` is appropriate for producing submissions, not for calculating accuracy without released ground truth.

In [ ]:
if RUN_TRAINING:
    summary = run_experiment(series, ['mlp', 'rbf'], ROOT / 'artifacts/dataset3', lookback=30, n_splits=5)
    display(summary.sort_values('rmse'))
else:
    print('Training skipped. Set RUN_TRAINING = True to run MLP and RBF.')

## 5. Published comparison

In [ ]:
published = paper_table().query("dataset == 'dataset_3'").pivot(index='model', columns='metric', values='value')
display(published[['mse', 'rmse', 'mae', 'r2', 'mape']].sort_values('rmse'))

## Interpretation and reporting note

LSTM has the lowest reported MSE, RMSE, MAE, and MAPE, while LSTM and RNN share the highest R². This differs from one statement in the paper abstract that names ARIMA-RBF as having the lowest Dataset 3 RMSE and MAE. The repository preserves the numerical tables and openly flags the inconsistency instead of silently changing either claim.